In [7]:

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [8]:
#create session
spark = ((SparkSession
            .builder
            .appName('suppliertransaction')
            .config('spark.jars', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
        ))

In [9]:
#loding data from source table 

tmp_suppliertransaction = (( spark
                                .read
                                .format('jdbc')
                                .option('url', mssqlsource)
                                .option('driver', mssqldriver)
                                .option('user', mssqluser)
                                .option('password', mssqlpwd)
                                .option('query', 
                                        """ 
                                            select
                                                
                                                supplierinvoicenumber,
                                                transactiontypeid,
                                                supplierid,
                                                purchaseorderid,
                                                suppliertransactionid,
                                                paymentmethodid,
                                                transactionamount,
                                                taxamount,
                                                amountexcludingtax,
                                                outstandingbalance,
                                                transactiondate,
                                                isfinalized
                                                
                                            from purchasing.suppliertransactions
                                            
                                        """)
                                )).load()

In [10]:
renamed_frame = (( tmp_suppliertransaction
                    .select('*', 
                            f.coalesce('supplierinvoicenumber', f.lit('999999'))
                                .alias('supplierinvoicenumber_n'),
                    
                            f.coalesce('purchaseorderid', f.lit('999999'))
                                .alias('purchaseorderid_n'),
                            
                            #substituting true-false for 0-1
                            f.when(tmp_suppliertransaction.isfinalized == 'true', 1)
                                .otherwise(0)
                                .alias('isfinalized_n')
                        )
                                
                ))

In [11]:
dropcol_transaction = (( renamed_frame
                            .drop('supplierinvoicenumber')
                            .drop('purchaseorderid')
                        #     .drop('isfinalized')
                            
                            #renaming
                            .withColumnRenamed('supplierinvoicenumber_n', 'suppliertransaction_number')
                            .withColumnRenamed('taxamount', 'tax')
                            .withColumnRenamed('amountexcludingtax', 'price')
                            .withColumnRenamed('transactionamount', 'total')
                            .withColumnRenamed('purchaseorderid_n', 'purchaseorderid')
                            .withColumnRenamed('outstandingbalance', 'balance')
                    ))

In [12]:
#organizing table
suppliertransaction_frame = ((  
                              dropcol_transaction
                                .select('suppliertransactionid',
                                        'transactiontypeid',
                                        'suppliertransaction_number',
                                        'purchaseorderid',
                                        'supplierid',
                                        'paymentmethodid',
                                        'price',
                                        'tax',
                                        'total',
                                        'balance',
                                        'transactiondate',
                                        'isfinalized')
                                ))

In [13]:
#write to datawarehouse
(( 
  suppliertransaction_frame
    .write
        .format('jdbc')
        .option('url', mssql_dwdest)
        .option('driver', mssqldriver)
        .option('user', mssqluser)
        .option('password', mssqlpwd)
        .option('dbtable', 'fact.suppliertransaction')
    .save(mode= 'overwrite')
  ))